In [3]:
import glob
import pandas as pd

# Load data

In [4]:
import unicodedata
import re
from unidecode import unidecode

def norm_text(text):
    if not isinstance(text, str):
        try:
            text = str(text)
        except:
            return ""

    text = unicodedata.normalize("NFC", text)
    text = unidecode(text)

    char_map = {
        "–": "-", "—": "--", "‘": "'", "’": "'", "“": '"', "”": '"', "…": "...",
        "•": "*", "·": ".", "×": "x", "÷": "/", "≤": "<=", "≥": ">=", "≠": "!=",
        "≈": "~", "∞": "inf", "∂": "d", "∫": "integral", "∑": "sum", "∏": "product",
        "√": "sqrt", "∝": "prop to", "∠": "angle", "△": "triangle", "□": "square",
        "∈": "in", "∉": "not in", "⊂": "subset", "⊃": "superset", "∪": "union",
        "∩": "intersect", "⊆": "subseteq", "⊇": "superseteq",
    }

    for char, replacement in char_map.items():
        text = text.replace(char, replacement)

    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)

    return text

In [39]:
def load_evidence(fn):
    ds_name = fn.split("/")[-3]
    df = pd.read_json(fn)
    # Unwrap the 'source' column (contains a dictionary) into separate columns
    df = df['source'].apply(pd.Series)
    df["ds"] = ds_name
    df["source_rationale"]  = df["source_rationale"].apply(lambda x: norm_text(x))
    return df

In [113]:
fns = glob.glob("../../data/*/evidence_human/evidence.json")
manuscript_fns = glob.glob("../../data/*/manuscript/manuscript.txt")


In [114]:
papers = {}

for m in manuscript_fns:
    m_id = m.split("/")[-3]
    with open(m, "r") as f:
        papers[m_id] = f.read()


In [115]:
papers.keys()

dict_keys(['kidney_Wu2018', 'testis_Shamis2020', 'yolksac_Goh2023', 'bone_He2021', 'adipose_Vijay2019', 'adipose_Emont2022', 'lung_Adams2020', 'heart_Tucker2020', 'liver_Guillams2022', 'bladder_Yu2019', 'pancreas_Segerstolpe2016', 'eye_Gautam2021', 'ovary_Wagner2020', 'placenta_Liu2018'])

In [149]:
dfs = []
for fn in fns:
    dfs.append(load_evidence(fn))
df = pd.concat(dfs)
tdf = df.query("source_type == 'text'").reset_index(drop=True)

In [150]:
import numpy as np

In [151]:
# missing papers
np.setdiff1d(tdf.ds.unique(), list(papers.keys()))

array(['adipose_Hildreth2021', 'adipose_Jaitin2019',
       'adipose_Massier2023', 'adipose_Merrick2019',
       'cortex_Booeshaghi2021'], dtype=object)

In [152]:
tdf["found_source_rationale"] = tdf.apply(lambda x: x["source_rationale"] in papers[x["ds"]] if x["ds"] in papers.keys() else False, axis=1)

In [153]:
agg = tdf.groupby("ds").agg({
    "found_source_rationale": ["count", "sum"], 
    }).rename(columns={"count": "total", "sum": "found"}, level=1)
agg[("found_source_rationale", "pct")] = 100*agg[("found_source_rationale", "found")] / agg[("found_source_rationale", "total")]
agg[("found_source_rationale", "complete")] = agg[("found_source_rationale", "pct")] == 100
agg["all_complete"] = agg[("found_source_rationale", "complete")]

In [154]:
# Add a total row at the bottom of the DataFrame
total_row = agg.sum(numeric_only=True)
total_row[("found_source_rationale", "pct")] = 100 * total_row[("found_source_rationale", "found")] / total_row[("found_source_rationale", "total")]


# Handling boolean columns for completeness
total_row[("found_source_rationale", "complete")] = total_row[("found_source_rationale", "pct")] == 100
total_row["all_complete"] = total_row[("found_source_rationale", "complete")]

# Add the total row to the DataFrame
agg.loc["total"] = total_row

# Sort the columns
agg = agg.sort_index(axis=1)

agg.reset_index()

/var/folders/89/1b_bcy3s39g4nkyxbdxvgzt40000gn/T/ipykernel_7405/2191187155.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'False' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  total_row[("found_source_rationale", "complete")] = total_row[("found_source_rationale", "pct")] == 100


ds all_complete found_source_rationale         \
                                                        complete  found   
0          adipose_Emont2022         True                   True   17.0   
1       adipose_Hildreth2021        False                  False    0.0   
2         adipose_Jaitin2019        False                  False    0.0   
3        adipose_Massier2023        False                  False    0.0   
4        adipose_Merrick2019        False                  False    0.0   
5          adipose_Vijay2019         True                   True   79.0   
6             bladder_Yu2019         True                   True   41.0   
7                bone_He2021         True                   True   72.0   
8      cortex_Booeshaghi2021        False                  False    0.0   
9             eye_Gautam2021         True                   True   25.0   
10          heart_Tucker2020         True                   True   73.0   
11             kidney_Wu2018         True                   True   31.0   
12        liver_Guillams2022         True                   True    5.0   
13            lung_Adams2020         True                   True   37.0   
14          ovary_Wagner2020         True                   True   65.0   
15  pancreas_Segerstolpe2016         True                   True  124.0   
16          placenta_Liu2018         True                   True  102.0   
17         testis_Shamis2020        False                  False   65.0   
18           yolksac_Goh2023         True                   True   96.0   
19                     total        False                  False  832.0   

                        
           pct   total  
0   100.000000    17.0  
1     0.000000    72.0  
2     0.000000    49.0  
3     0.000000    29.0  
4     0.000000    29.0  
5   100.000000    79.0  
6   100.000000    41.0  
7   100.000000    72.0  
8     0.000000     4.0  
9   100.000000    25.0  
10  100.000000    73.0  
11  100.000000    31.0  
12  100.000000     5.0  
13  100.000000    37.0  
14  100.000000    65.0  
15  100.000000   124.0  
16  100.000000   102.0  
17   56.034483   116.0  
18  100.000000    96.0  
19   78.048780  1066.0

In [155]:
# find specific datasets with missing source rationales
ds = 'adipose_Vijay2019'
tdf.query(f"ds == '{ds}' & (~found_source_rationale)")[["source_rationale"]].drop_duplicates()

,source_rationale
